# ColabUMA — UMAによる分子構造最適化

Metaの機械学習ポテンシャル **UMA** で分子の構造最適化を行うノートブックです。コードを書く必要はありません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. 入力フォーム** で出力名、電荷、スピン多重度、収束判定を指定する
4. **ランタイム → すべて実行** を押す
5. **2. ファイル選択と事前確認** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。あとは放置できます
6. 一番下のセルで 3D 表示・エネルギー・収束グラフを確認し、結果のZIPをダウンロードする

gjfからは構造だけを読み込みます。電荷と多重度はフォームの値を使います。ZIPの中の `_opt.log` は最適化の経路をGaussianの出力形式で書いたもので、GaussViewで開くと各ステップの構造とエネルギー変化を確認できます（Gaussianで計算した結果ではありません）。

2回目以降は手順3〜6を繰り返すだけです（インストールとモデル読み込みは自動でスキップされます）。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 注意事項

- 最適化は鞍点に収束することがあります。重要な構造は **振動計算** をオンにして確認してください。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。最終的なエネルギー比較はDFTで確認してください。
- 小さな分子（数十原子以下）では、DFTを直接使う方が確実な場面が多くあります。
- 周期系（結晶・表面）の最適化には対応していません。
- 結合長・角度・二面角のスキャンは [ColabUMA_Scan](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_Scan.ipynb)、遷移状態の探索は [ColabUMA_TS](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_TS.ipynb) を使ってください。最適化 → スキャン → TS を同じセッションで続けるときは [ColabUMA_Workflow](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_Workflow.ipynb) を使うと、インストールとモデル読み込みが1回で済みます。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabUMA/blob/main/LICENSE)
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabUMA#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。次のセルで xyz または gjf ファイルを選ぶと、あとは放置できます。

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

#@markdown ### 構造
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### 計算条件
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown 収束判定 fmax（eV/Å）。normal は Gaussian の既定、tight は Gaussian の opt=tight と同じくらいです。
convergence = "loose（0.05）"  #@param ["loose（0.05）", "normal（0.02）", "tight（0.001）", "数値指定"]
#@markdown 「数値指定」を選んだときの fmax（eV/Å）。
fmax_custom = 0.05  #@param {type:"number"}
max_steps = 500  #@param {type:"integer"}
run_vibrations = False  #@param {type:"boolean"}

In [ ]:
#@title 2. ファイル選択と事前確認 { display-mode: "form" }
#@markdown ファイルの選択とトークンの確認をここで済ませます。このあとのインストールと計算は放置できます。

import os
import subprocess
import sys

COLABUMA_REF = "main"  # 共通コード（colabuma パッケージ）を取ってくる GitHub のブランチ


def _use_colabuma(ref):
    # ノートと同じフォルダに colabuma があればそれを、なければ GitHub から取ってきて使う
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabUMA" if os.path.isdir("/content") else os.path.expanduser("~/ColabUMA")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabUMA.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"ColabUMA の共通コードを GitHub から取得できませんでした（ブランチ {ref}）。\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import env, opt, output, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
ok("入力の確認が終わりました。ここからは放置できます。")

In [ ]:
#@title 3. セットアップ（インストール・認証） { display-mode: "form" }
DEVICE = env.setup()

In [ ]:
#@title 4. モデル読み込み { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. 構造の準備 { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"{atoms0.get_chemical_formula()}（{len(atoms0)} 原子）の初期構造を用意しました。")

In [ ]:
#@title 6. 最適化 { display-mode: "form" }
#@markdown 「収束」のバーは、fmax が最初の値から収束判定までに桁でどこまで近づいたかを示します。途中で fmax が上がるとバーも戻ります。
print(f"{optimizer_name} で最適化します（fmax < {fmax} eV/Å、最大 {max_steps} ステップ）")
best = opt.optimize(atoms0, calc, job, "構造")
vib_result = opt.vibrations(best["atoms"], calc) if run_vibrations else None

In [ ]:
#@title 7. 結果表示と保存 { display-mode: "form" }
files, log = opt.report(best, vib_result, job)
files["log.txt"] = output.log_text("ColabUMA 計算ログ", source_desc, job, [log])
zip_path = output.save_zip(job.out_prefix, files)